Import Library

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset, random_split
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from PIL import Image
import json
import os
import random
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import gdown
import re 
import zipfile
from tqdm.auto import tqdm 

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        # Dành cho Mac
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print(f"Using device: {device}")

Using device: mps


In [ ]:
DRIVE_FOLDER_LINK = 'https://drive.google.com/drive/folders/1xGIkrz0BthCz7NHw8OUuA12Joh915K8E?usp=share_link'

def setup_data(drive_link):
    # Tạo thư mục chứa data
    base_dir = os.getcwd()
    data_dir = os.path.join(base_dir, 'data')
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)
    
    print(f"Data saved at: {data_dir}")

    # Lấy ID từ Link
    try:
        folder_id = drive_link.split('/')[-1].split('?')[0]
    except:
        return None, None
    
    # Download Folder
    print("Downloading...")
    try:
        gdown.download_folder(id=folder_id, output=data_dir, quiet=False, use_cookies=False)
        print("Download successfully")
    except Exception as e:
        print(f"Error: {e}")
        return None, None

    #Giải nén
    img_root = os.path.join(data_dir, 'images')
    if not os.path.exists(img_root):
        os.makedirs(img_root)

    for file in os.listdir(data_dir):
        if file.endswith('.zip'):
            print(f"Extracting {file}...")
            try:
                with zipfile.ZipFile(os.path.join(data_dir, file), 'r') as zip_ref:
                    zip_ref.extractall(img_root)
            except:
                pass 

    # Xác định folder ảnh chính xác
    if os.path.exists(os.path.join(img_root, 'val2014')):
        img_dir = os.path.join(img_root, 'val2014')
    else:
        img_dir = img_root

    # Tìm file JSON
    json_path = None
    for root, dirs, files in os.walk(data_dir):
        for file in files:
            if file.endswith('.json'): 
                if 'subset' in file or 'questions' in file or 'annotation' in file:
                    json_path = os.path.join(root, file)
                
    return img_dir, json_path

IMG_DIR, JSON_FILE = setup_data(DRIVE_FOLDER_LINK)

print("-" * 30)
if IMG_DIR and JSON_FILE:
    print(f"Training...")
    print(f"Ảnh: {IMG_DIR}")
    print(f"JSON: {JSON_FILE}")
else:
    print("Please try again")
print("-" * 30)

Data saved at: /Users/quangvo/Library/Mobile Documents/com~apple~CloudDocs/juniorYear/DeepLearning/Midterm_523H0173_523H0178/DL_VQA_Project/data
Downloading...


Retrieving folder contents


Processing file 1iTgngqAEuK_SPWjmlz_el4yfO8V-bUTm v2_OpenEnded_mscoco_val2014_questions.json
Processing file 1dqS3utsweJxnxYqKQB1-rnrnXpf8KoYp val2014.zip


Retrieving folder contents completed
Building directory structure
Building directory structure completed

Downloading...
From: https://drive.google.com/uc?id=1iTgngqAEuK_SPWjmlz_el4yfO8V-bUTm
To: /Users/quangvo/Library/Mobile Documents/com~apple~CloudDocs/juniorYear/DeepLearning/Midterm_523H0173_523H0178/DL_VQA_Project/data/v2_OpenEnded_mscoco_val2014_questions.json






































































100%|██████████| 20.2M/20.2M [00:06<00:00, 3.12MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1dqS3utsweJxnxYqKQB1-rnrnXpf8KoYp
From (redirected): https://drive.google.com/uc?id=1dqS3utsweJxnxYqKQB1-rnrnXpf8KoYp&confirm=t&uuid=fe11d0e9-a927-4096-833f-862b5fcd89d2
To: /Users/quangvo/Library/Mobile Documents/com~apple~CloudDocs/juniorYear/DeepLearning/Midterm_523H0173_523H0178/DL_VQA_Project/data/val2014.zip



































































































































In [4]:
class Vocabulary:
    def __init__(self):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.freq_threshold = 1

    def __len__(self): return len(self.itos)

    @staticmethod
    def tokenizer(text):
        # Chuyển về chữ thường, bỏ dấu câu cơ bản
        return str(text).lower().replace('?', '').replace('.', '').replace(',', '').split()

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 4
        for sentence in sentence_list:
            for word in self.tokenizer(sentence):
                frequencies[word] += 1
        
        # Chỉ thêm từ xuất hiện nhiều hơn threshold
        for word, count in frequencies.items():
            if count >= self.freq_threshold:
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1
    
    def numericalize(self, text):
        tokenized_text = self.tokenizer(text)
        return [
            self.stoi.get(token, self.stoi["<UNK>"])
            for token in tokenized_text
        ]

In [5]:
class VQADataset:
    def __init__(self, json_path, img_dir, vocab=None, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        
        # Load dữ liệu JSON
        print(f"Processing JSON...: {os.path.basename(json_path)}...")
        with open(json_path, 'r') as f:
            data_raw = json.load(f)
            # Tùy format JSON mà key chứa câu hỏi có thể khác nhau
            if 'questions' in data_raw:
                self.data = data_raw['questions']
            elif 'annotations' in data_raw:
                self.data = data_raw['annotations']
            else:
                self.data = data_raw

        # Tự động xây dựng từ điển nếu chưa có
        if vocab is None:
            print("Processing Vocab...")
            self.vocab = Vocabulary()
            # Gom toàn bộ câu hỏi để học từ vựng
            all_questions = [item.get('question', '') for item in self.data]
            self.vocab.build_vocabulary(all_questions)
            print(f"Vocab size: {len(self.vocab)} words.")
        else:
            self.vocab = vocab

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        item = self.data[index]
        
        # --- 1. XỬ LÝ ẢNH ---
        # Format tên file ảnh COCO: COCO_val2014_000000xxxxxx.jpg
        img_id = item['image_id']
        img_filename = f"COCO_val2014_{int(img_id):012d}.jpg"
        img_path = os.path.join(self.img_dir, img_filename)
        
        try:
            image = Image.open(img_path).convert("RGB")
        except FileNotFoundError:
            # Fallback: Tạo ảnh đen nếu thiếu file (tránh crash)
            image = Image.new('RGB', (224, 224))
            
        if self.transform:
            image = self.transform(image)

        # --- 2. XỬ LÝ CÂU HỎI ---
        question = item.get('question', '')
        # Chuyển text -> List số (Tensor)
        q_vec = [self.vocab.stoi["<SOS>"]] + self.vocab.numericalize(question) + [self.vocab.stoi["<EOS>"]]
        
        return image, torch.tensor(q_vec)

In [6]:
def collate_fn(batch):
    images, questions = zip(*batch)

    # Stack ảnh thành khối: [Batch, 3, 224, 224]
    images = torch.stack(images, 0)
    
    # Padding câu hỏi cho bằng độ dài nhau: [Batch, Max_Len]
    questions = pad_sequence(questions, batch_first=True, padding_value=0)
    
    return images, questions

  2%|▏         | 128M/6.62G [01:36<17:25:20, 103kB/s]

In [1]:
BATCH_SIZE = 32

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

if 'IMG_DIR' in globals() and 'JSON_FILE' in globals():
    #Load toàn bộ dữ liệu
    full_dataset = VQADataset(JSON_FILE, IMG_DIR, transform=transform)
    
    #(80% Train - 10% Val - 10% Test)
    total_count = len(full_dataset)
    train_size = int(0.8 * total_count)
    val_size = int(0.1 * total_count)
    test_size = total_count - train_size - val_size
    
    #Chia dataset
    train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)

    # 4. In thông tin
    print("-" * 30)
    print(f"Vocab Size: {len(full_dataset.vocab)}")
    print(f"Total Data: {len(full_dataset)}")
    print(f"Train Set : {len(train_dataset)}")
    print(f"Val Set   : {len(val_dataset)}")
    print(f"Test Set  : {len(test_dataset)}")

NameError: name 'transforms' is not defined